### This notebook shall be used to transform the raw DGCA data into a processed data with established hierarchy levels that will enable forecasting and reconciliation. The expected output is a long format dataset without any missing values and with three levels of hierarchy

In [2]:
# Importing the necessary libraries
import pandas as pd
import numpy as np

In [7]:
# Loading the data file
df = pd.read_csv('../data/raw/city_domestic.csv')

In [12]:
df.shape

(63173, 13)

In [9]:
# Adding a date column
df['date'] = pd.to_datetime(df[['Year', 'Month']].assign(DAY=1))

In [13]:
print("Date Range: ",df['date'].min(), " to ",df['date'].max())

Date Range:  2015-04-01 00:00:00  to  2026-02-01 00:00:00


In [11]:
# Directional Route Label
df['route'] = df['City1'] + " - " + df['City2']

In [15]:
# Check for directional setup
sample = df[(df['City1'] == 'DELHI') & (df['City2'] == "MUMBAI")][['date','PaxToCity2','PaxFromCity2']].head()
print(sample)

            date  PaxToCity2  PaxFromCity2
32431 2019-05-01    295629.0      291304.0
32432 2019-06-01    298329.0      275226.0
32433 2019-07-01    284412.0      285976.0
32434 2019-08-01    294305.0      299669.0
32435 2019-09-01    291390.0      298080.0


In [16]:
# Check for directional setup
sample = df[(df['City1'] == 'MUMBAI') & (df['City2'] == "DELHI")][['date','PaxToCity2','PaxFromCity2']].head()
print(sample)

            date  PaxToCity2  PaxFromCity2
56715 2015-04-01    255579.0      231679.0
56716 2015-05-01    264917.0      270727.0
56717 2015-06-01    214142.0      248419.0
56718 2015-07-01    235638.0      234464.0
56719 2015-08-01    235092.0      236913.0


In [17]:
# Defining the target variable
df['y'] = df['PaxToCity2']

In [18]:
# Aggregating the total volume and month count per route
route_stats = df.groupby('route').agg(
    total_volume = ('y','sum'),
    month_count = ('Month','count')
).reset_index()

In [31]:
# Applying coverage filter: Selecting only those routes for which we have data for at least 60 months
route_stats_filtered = route_stats[route_stats['month_count']>=60]

In [ ]:
print(f"Routes after coverage filter (data available for minimum 60 months): {len(route_stats_filtered)}")

Routes after coverage filter (data available for minimum 100 months): 411


In [28]:
# Applying volume filter: Selecting the top 200 routes by volume
top_routes = route_stats_filtered.nlargest(200,'total_volume')['route'].tolist()

In [29]:
print(f"Final selected routes: {len(top_routes)}")

Final selected routes: 200


In [35]:
selected_volume = route_stats_filtered[route_stats_filtered['route'].isin(top_routes)]['total_volume'].sum()
total_volume_sum = df['y'].sum()
print(f"200 selected routes cover {(selected_volume/total_volume_sum)*100:0.1f}% of directional volume")

200 selected routes cover 78.9% of directional volume


In [36]:
# Building the route level series
df_routes = df[df['route'].isin(top_routes)].copy() # Filtering the top routes

# Route level series (one row per route per month)
route_level = df_routes.groupby(['City1', 'City2', 'date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
route_level['unique_id'] = 'TOTAL/' + route_level['City1'] + '/' + route_level['City1'] + '-' + route_level['City2']

# Dropping all unnecessary columns, renaming date to ds
route_level = route_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("Route-level Series:",route_level['unique_id'].nunique(), " routes")
print(route_level.head())


Route-level Series: 200  routes
                          unique_id         ds        y
0  TOTAL/AGARTALA/AGARTALA-GUWAHATI 2019-05-01  11338.0
1  TOTAL/AGARTALA/AGARTALA-GUWAHATI 2019-06-01  10786.0
2  TOTAL/AGARTALA/AGARTALA-GUWAHATI 2019-07-01  11355.0
3  TOTAL/AGARTALA/AGARTALA-GUWAHATI 2019-08-01  10927.0
4  TOTAL/AGARTALA/AGARTALA-GUWAHATI 2019-09-01   9271.0


In [37]:
# Building the airport level series (sum of all outbound traffic from origin airport, per month)
airport_level = df_routes.groupby(['City1','date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
airport_level['unique_id'] = 'TOTAL/' + airport_level['City1']

# Dropping all unnecessary columns and renaming date to ds
airport_level = airport_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("Airport-level Series:",airport_level['unique_id'].nunique(), 'airports')
print(airport_level.head())

Airport-level Series: 31 airports
        unique_id         ds        y
0  TOTAL/AGARTALA 2019-05-01  47289.0
1  TOTAL/AGARTALA 2019-06-01  48748.0
2  TOTAL/AGARTALA 2019-07-01  51463.0
3  TOTAL/AGARTALA 2019-08-01  50545.0
4  TOTAL/AGARTALA 2019-09-01  44051.0


In [40]:
# Building the national level series
total_level = df_routes.groupby(['date'])['y'].sum().reset_index()

# Unique ID for the hierarchy
total_level['unique_id'] = 'TOTAL'

# Dropping all unnecessary columns and renaming date as ds
total_level = total_level[['unique_id','date','y']].rename(columns = {'date':'ds'})

print("National level series:",total_level['unique_id'].nunique(), "series")
print(total_level.head())

print("Number of months:",len(total_level))

National level series: 1 series
  unique_id         ds          y
0     TOTAL 2015-04-01  2364483.0
1     TOTAL 2015-05-01  2547864.0
2     TOTAL 2015-06-01  2327666.0
3     TOTAL 2015-07-01  2431149.0
4     TOTAL 2015-08-01  2381990.0
Number of months: 129


In [41]:
# Stacking all three hierarchical levels together
hierarchy_df = pd.concat([total_level,airport_level,route_level], ignore_index = True)


# Sorting the df
hierarchy_df = hierarchy_df.sort_values(['unique_id','ds']).reset_index(drop = True)

print("Combined hierrachy dataset:")
print('Total rows: ',len(hierarchy_df))
print("Total unique series: ",hierarchy_df['unique_id'].nunique())

print("\nSeries count by level:")
print("TOTAL",(hierarchy_df['unique_id']=='TOTAL').sum() > 0)
print("Series Breakdown: ",hierarchy_df['unique_id'].str.count('/').value_counts().to_dict())

Combined hierrachy dataset:
Total rows:  25470
Total unique series:  232

Series count by level:
TOTAL True
Series Breakdown:  {2: 21782, 1: 3559, 0: 129}


In [42]:
# Handling missing values
def complete_series(group):
    # Set date index and resample to monthly start frequency
    g = group.set_index('ds').asfreq('MS')

    # Interpolate only internal gaps (limit_area = 'inside' doesn't fill before first or after last real value)
    g['y'] = g['y'].interpolate(method = 'linear', limit_area = 'inside')
    g['unique_id'] = group['unique_id'].iloc[0]
    return g.reset_index()

hierarchy_complete = hierarchy_df.groupby('unique_id',group_keys = False).apply(complete_series)

print("Before: ",len(hierarchy_df), " rows")
print("After: ",len(hierarchy_complete), " rows")

print("Remaining null values in y: ",hierarchy_complete['y'].isna().sum())

Before:  25470  rows
After:  27077  rows
Remaining null values in y:  0


/var/folders/nq/rpmm4zk552n5ksgcbp8hz7j80000gn/T/ipykernel_48984/3737795510.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hierarchy_complete = hierarchy_df.groupby('unique_id',group_keys = False).apply(complete_series)


In [43]:
# Coherence Check

check_month = '2025-12-01'

# Check 1: Do all airport series sum to TOTAL?
total_val = hierarchy_complete[
    (hierarchy_complete['unique_id'] == 'TOTAL') &
    (hierarchy_complete['ds'] == check_month)]['y'].values[0]

airport_sum = hierarchy_complete[
    (hierarchy_complete['unique_id'].str.count('/')==1) & 
    (hierarchy_complete['ds'] == check_month)]['y'].sum()

print(f"Check Month: {check_month}")
print(f"Total series value: {total_val}")
print(f"Sum of all airport series: {airport_sum}")
print(f"Match: {abs(total_val - airport_sum) < 1}")

Check Month: 2025-12-01
Total series value: 5664328.0
Sum of all airport series: 5664328.0
Match: True


In [45]:
# Check 2: Do all route series sum to the origin airport?
check_month = '2025-12-01'
check_airport = 'MUMBAI'

# Airport level value
airport_val = hierarchy_complete[
    (hierarchy_complete['unique_id'] == f'TOTAL/{check_airport}') &
    (hierarchy_complete['ds'] == check_month)]['y'].values[0]

# Sum of all routes originating from the check airport
routes_sum = hierarchy_complete[
    (hierarchy_complete['unique_id'].str.startswith(f'TOTAL/{check_airport}/')) &
    (hierarchy_complete['ds'] == check_month)]['y'].sum()

print(f"Airport: {check_airport} & Month: {check_month}")
print(f"Airport series value: {airport_val}")
print(f"Sum of all route series: {routes_sum}")
print(f"Match: {abs(airport_val - routes_sum) < 1}")

Airport: MUMBAI & Month: 2025-12-01
Airport series value: 185506.0
Sum of all route series: 185506.0
Match: True


In [ ]:
# 